In [ ]:
from IPython.display import display
import os
import cv2
import torch
import torch.nn as nn
from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
clean_hf_dir = '/content/drive/MyDrive/rainy-image-dataset/train/clean'
test_rain_dir='/content/drive/MyDrive/rainy-image-dataset/test/rainy'
test_clean_dir='/content/drive/MyDrive/rainy-image-dataset/test/clean'
hf_dir = '/content/drive/MyDrive/HF'
lf_dir = '/content/drive/MyDrive/LF'

print("\nRainy HF images:")
print(sorted(os.listdir(hf_dir)))
print("\nRainy LF images:")
print(sorted(os.listdir(lf_dir)))
print("\nTest Rain images:")
print(sorted(os.listdir(test_rain_dir)))
print("\nTest Clean images:")
print(sorted(os.listdir(test_clean_dir)))
print("\nClean images:")
print(sorted(os.listdir(clean_hf_dir)))

In [ ]:
##################Dataset for LF & HF Image Pairs
class RainDetailDataset(Dataset):
    def __init__(self, lf_dir, hf_dir, clean_hf_dir, size=(256, 256)):
        self.lf_dir = lf_dir
        self.hf_dir = hf_dir
        self.clean_hf_dir = clean_hf_dir
        self.names = sorted(os.listdir(hf_dir))
        self.size = size

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]

        # Construct full paths
        lf_path = os.path.join(self.lf_dir, name)
        hf_path = os.path.join(self.hf_dir, name)
        clean_hf_path = os.path.join(self.clean_hf_dir, name)
        print(clean_hf_path)  # Debugging print statement

        # Load with cv2
        lf = cv2.imread(lf_path)
        hf = cv2.imread(hf_path)
        clean_hf = cv2.imread(clean_hf_path)

        # Handle cases where images are not loaded
        if lf is None:
            raise FileNotFoundError(f"LF image not found: {lf_path}")
        if hf is None:
            raise FileNotFoundError(f"HF image not found: {hf_path}")
        if clean_hf is None:
            raise FileNotFoundError(f"Clean HF image not found: {clean_hf_path}")

        # Resize images
        lf = cv2.resize(lf, self.size)
        hf = cv2.resize(hf, self.size)
        clean_hf = cv2.resize(clean_hf, self.size)

        lf = cv2.cvtColor(lf, cv2.COLOR_BGR2RGB)
        hf = cv2.cvtColor(hf, cv2.COLOR_BGR2RGB)
        clean_hf = cv2.cvtColor(clean_hf, cv2.COLOR_BGR2RGB)

        # To tensor
        lf = torch.from_numpy(lf.transpose((2, 0, 1))).float() / 255.0
        hf = torch.from_numpy(hf.transpose((2, 0, 1))).float() / 255.0
        clean_hf = torch.from_numpy(clean_hf.transpose((2, 0, 1))).float() / 255.0

        return hf, clean_hf, lf



In [ ]:
class RainDataset(Dataset):
    def __init__(self, rainy_dir, clean_dir, size=(256, 256)):
        self.rainy_dir = rainy_dir
        self.clean_dir = clean_dir
        self.clean_images = os.listdir(clean_dir)
        self.size = size

    def __len__(self):
        return len(self.clean_images)

    def __getitem__(self, idx):
        clean_filename = self.clean_images[idx]
        clean_path = os.path.join(self.clean_dir, clean_filename)
        rainy_filename = f"{os.path.splitext(clean_filename)[0]}_1.jpg"
        rainy_path = os.path.join(self.rainy_dir, rainy_filename)

        # Read using cv2 and normalize
        clean = cv2.imread(clean_path)
        rainy = cv2.imread(rainy_path)
        clean = cv2.resize(clean, self.size)
        rainy = cv2.resize(rainy, self.size)
        clean = cv2.cvtColor(clean, cv2.COLOR_BGR2RGB)
        rainy = cv2.cvtColor(rainy, cv2.COLOR_BGR2RGB)

        # Normalize to [0,1] and convert to tensor
        clean = torch.from_numpy(clean.transpose((2, 0, 1))).float() / 255.0
        rainy = torch.from_numpy(rainy.transpose((2, 0, 1))).float() / 255.0

        return rainy, clean


In [ ]:
# Dataset and DataLoader
train_dataset = RainDetailDataset(lf_dir, hf_dir, clean_hf_dir)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [ ]:
class DerainProcessorCV2:
    def __init__(self, radius=15, eps=1e-6):
        self.radius = radius
        self.eps = eps

    def guided_filter(self, I, p, r, eps):
        return cv2.ximgproc.guidedFilter(
            guide=I, src=p, radius=r, eps=eps, dDepth=-1
        )

    def decompose(self, img_tensor):
        img_np = img_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
        img_np = np.clip(img_np, 0, 1).astype(np.float32)

        base = np.zeros_like(img_np)
        for c in range(3):
            base[:, :, c] = self.guided_filter(img_np[:, :, c], img_np[:, :, c], self.radius, self.eps)

        detail = img_np - base
        base = torch.from_numpy(base).permute(2, 0, 1).unsqueeze(0).float()
        detail = torch.from_numpy(detail).permute(2, 0, 1).unsqueeze(0).float()
        return base, detail

    def reconstruct(self, base, predicted_detail, enhance=True):
        if enhance:
            predicted_detail *= 2
        return torch.clamp(base + predicted_detail, 0, 1)


In [ ]:
pip install opencv-contrib-python


In [ ]:
###Derain net class###
import torch
import torch.nn as nn

class DerainNet(nn.Module):
    def __init__(self):
        super(DerainNet, self).__init__()

        # Encoder
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # Bottleneck
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # Decoder
        self.deconv1 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.deconv2 = nn.Sequential(
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Sigmoid()  # Because output should be in [0,1]
        )

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)
        x4 = self.deconv1(x3)
        out = self.deconv2(x4)
        return out


In [ ]:
!pip install pytorch-ssim

In [ ]:
##code for simple_ssim since we are not installing the library
import torch
import torch.nn.functional as F

def simple_ssim(img1, img2, window_size=11, size_average=True):
    # Constants for stability
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    mu1 = F.avg_pool2d(img1, window_size, stride=1, padding=window_size//2)
    mu2 = F.avg_pool2d(img2, window_size, stride=1, padding=window_size//2)

    mu1_sq = mu1 * mu1
    mu2_sq = mu2 * mu2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.avg_pool2d(img1 * img1, window_size, stride=1, padding=window_size//2) - mu1_sq
    sigma2_sq = F.avg_pool2d(img2 * img2, window_size, stride=1, padding=window_size//2) - mu2_sq
    sigma12 = F.avg_pool2d(img1 * img2, window_size, stride=1, padding=window_size//2) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

    if size_average:
        return ssim_map.mean()
    else:
        return ssim_map


In [ ]:
####Preprocessing step -- to remove noise in the input image
import torchvision.transforms as T

# Simple denoising / smoothing
def preprocess_rainy_input(rainy_input_batch):
    """
    rainy_input_batch: torch tensor (batch_size, 3, 256, 256)
    Applies light Gaussian blur to reduce noise.
    """
    blur = T.GaussianBlur(kernel_size=5, sigma=(0.5, 1.5))
    return blur(rainy_input_batch)


In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Dataset class
class DerainDetailDataset(Dataset):
    def __init__(self, hf_dir, clean_hf_dir, lf_dir, size=(256, 256)):
        self.hf_dir = hf_dir
        self.clean_hf_dir = clean_hf_dir
        self.lf_dir = lf_dir
        self.names = sorted(os.listdir(hf_dir))
        self.size = size

        hf_names = set(os.listdir(hf_dir))
        lf_names = set(os.listdir(lf_dir))
        clean_hf_names = set(os.listdir(clean_hf_dir))

        # Only keep names that exist in all 3
        self.names = sorted(list(hf_names & lf_names & clean_hf_names))

        if len(self.names) == 0:
            raise ValueError("No matching images in all three folders.")

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]

        hf_path = os.path.join(self.hf_dir, name)
        clean_hf_path = os.path.join(self.clean_hf_dir, name)
        lf_path = os.path.join(self.lf_dir, name)

        hf = cv2.imread(hf_path)
        clean_hf = cv2.imread(clean_hf_path)
        lf = cv2.imread(lf_path)

        # Check if any failed to load
        if hf is None or clean_hf is None or lf is None:
            raise FileNotFoundError(f"Failed to load image: {hf_path}, {clean_hf_path}, or {lf_path}")

        hf = cv2.resize(hf, self.size)
        clean_hf = cv2.resize(clean_hf, self.size)
        lf = cv2.resize(lf, self.size)

        hf = cv2.cvtColor(hf, cv2.COLOR_BGR2RGB)
        clean_hf = cv2.cvtColor(clean_hf, cv2.COLOR_BGR2RGB)
        lf = cv2.cvtColor(lf, cv2.COLOR_BGR2RGB)

        hf = torch.from_numpy(hf.transpose((2, 0, 1))).float() / 255.0
        clean_hf = torch.from_numpy(clean_hf.transpose((2, 0, 1))).float() / 255.0
        lf = torch.from_numpy(lf.transpose((2, 0, 1))).float() / 255.0

        return hf, clean_hf, lf

# DerainNet model
import pytorch_ssim

class DerainNet(nn.Module):
    def __init__(self):
        super(DerainNet, self).__init__()

        # Encoder
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # Bottleneck
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # Decoder
        self.deconv1 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.deconv2 = nn.Sequential(
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Sigmoid()  # Because output should be in [0,1]
        )

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)
        x4 = self.deconv1(x3)
        out = self.deconv2(x4)
        return out


# Set paths
hf_dir = '/content/drive/MyDrive/HF'
clean_hf_dir = '/content/drive/MyDrive/rainy-image-dataset/train/clean'
lf_dir = '/content/drive/MyDrive/LF'



# Define Combined Loss
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.8):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        #self.ssim = pytorch_ssim.SSIM()

    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        #ssim_loss = 1 - ssim(pred.cpu().detach().numpy(), target.cpu().detach().numpy())  #Convert to numpy for SSIM computation
        #ssim_loss = 1 - torch_ssim(pred, target, data_range=1.0, size_average=True)
        ssim_loss = 1 - simple_ssim(pred, target)
        return self.alpha * mse_loss + (1 - self.alpha) * ssim_loss

# Set up
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DerainNet().to(device)
criterion = CombinedLoss(alpha=0.8)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)




# Dataset
#train_dataset = DerainDetailDataset(hf_dir, clean_hf_dir, lf_dir)
#train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)


In [ ]:
######Use random_split for 80-20 split:##########333
from torch.utils.data import random_split


###Full dataset###
full_dataset = DerainDetailDataset(hf_dir, clean_hf_dir, lf_dir)

# 80-20 split
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
test_size = total_size - train_size

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

### Create DataLoaders########
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)



In [ ]:
##installing libraries for perceptual loss
!pip install torchmetrics

In [ ]:
!pip install pytorch-msssim


In [ ]:
##Defining all loss before training loop
import torch.nn as nn
import torch.nn.functional as F
from pytorch_msssim import ssim  # If using pytorch_msssim package
from torchvision import models

# MSE loss
mse_loss_fn = nn.MSELoss()

# SSIM loss (using pytorch_msssim)
def ssim_loss_fn(pred, target):
    return ssim(pred, target, data_range=1.0, size_average=True)

# Perceptual loss (based on VGG16)
class PerceptualLoss(nn.Module):
    def __init__(self, resize=True):
        super(PerceptualLoss, self).__init__()
        vgg = models.vgg16(pretrained=True).features[:16].eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg
        self.resize = resize

    def forward(self, pred, target):
        if self.resize:
            pred = F.interpolate(pred, size=(224, 224), mode='bilinear', align_corners=False)
            target = F.interpolate(target, size=(224, 224), mode='bilinear', align_corners=False)
        pred_features = self.vgg(pred)
        target_features = self.vgg(target)
        return F.l1_loss(pred_features, target_features)

perceptual_loss_fn = PerceptualLoss().to(device)


In [ ]:
# Set up training for 70 batches
import matplotlib.pyplot as plt  # Importing Matplotlib for visualization
import numpy as np
import torch
import os

# Create the output_images directory if it doesn't exist, for saving the output images
os.makedirs("./output_images", exist_ok=True)

# Set up training for 70 batches
num_batches_to_run = 70

model.train()
epoch_loss = 0.0
print("\n▶️  Running 70 Batches Only")

alpha = 0.8  # You can tune this between 0.8 - 0.9

for batch_idx, (hf_input, clean_hf_target, lf_input) in enumerate(train_loader):
    if batch_idx >= num_batches_to_run:
        print("✅ Reached 70 batches.")
        break

    hf_input = hf_input.to(device)
    clean_hf_target = clean_hf_target.to(device)

    #  Apply preprocessing
    hf_input = preprocess_rainy_input(hf_input)

    ## Forward pass
    predicted_hf = model(hf_input)* 0.5  # [5] scaled down prediction


    # Match shapes if needed (ensuring that both predicted and target images are the same size)
    min_h = min(predicted_hf.shape[2], clean_hf_target.shape[2], lf_input.shape[2])
    min_w = min(predicted_hf.shape[3], clean_hf_target.shape[3], lf_input.shape[3])
    predicted_hf = predicted_hf[:, :, :min_h, :min_w]
    clean_hf_target = clean_hf_target[:, :, :min_h, :min_w]
    lf_input = lf_input[:, :, :min_h, :min_w]

     # Combine predicted HF and LF to get final output
    final_output = predicted_hf + lf_input  # No alpha scaling needed
    final_output = torch.clamp(final_output, 0.0, 1.0)  # Ensure pixel values are between 0 and 1

     #  Prepare clean full image (target) for loss
    clean_full_target = clean_hf_target + lf_input
    clean_full_target = torch.clamp(clean_full_target, 0.0, 1.0)


     #Ensure correct input naming and detachment of tensors
    clean_hf = clean_hf_target  # Just aliasing for clarity
    #loss = loss_mse + 0.1 * loss_perceptual + 0.5 * loss_ssim
    loss_mse = mse_loss_fn(predicted_hf, clean_hf)
    loss_ssim = 1 - ssim_loss_fn(predicted_hf, clean_hf)
    loss_perceptual = perceptual_loss_fn(predicted_hf.detach(), clean_hf.detach())

    # Final combined loss
    loss = loss_mse + 0.1 * loss_perceptual + 0.5 * loss_ssim

      # Backpropagation
    optimizer.zero_grad()
    loss.backward(retain_graph=True)
    optimizer.step()

    epoch_loss += loss.item()

    # Optional progress print
    if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == num_batches_to_run:
        print(f"  Batch {batch_idx+1}/{num_batches_to_run} - Loss: {loss.item():.4f}")

    # Visualization after processing each batch
    if (batch_idx + 1) % 10 == 0:
        with torch.no_grad():
            # Prepare hf_input (Rainy input)
            hf_input_np = hf_input.cpu().detach().numpy()
            hf_input_np = np.transpose(hf_input_np, (0, 2, 3, 1))
            hf_input_np = np.clip(hf_input_np * 255, 0, 255).astype(np.uint8)

            # Prepare clean_hf_target (Ground truth clean)
            clean_hf_target_np = clean_hf_target.cpu().detach().numpy()
            clean_hf_target_np = np.transpose(clean_hf_target_np, (0, 2, 3, 1))
            clean_hf_target_np = np.clip(clean_hf_target_np * 255, 0, 255).astype(np.uint8)

            # Prepare predicted_hf (Predicted clean)
            predicted_hf_np = predicted_hf.cpu().detach().numpy()
            predicted_hf_np = np.transpose(predicted_hf_np, (0, 2, 3, 1))
            predicted_hf_np = np.clip(predicted_hf_np * 255, 0, 255).astype(np.uint8)

            #predictted final output
            final_output_np = final_output.cpu().detach().numpy()
            final_output_np = np.transpose(final_output_np, (0, 2, 3, 1))
            final_output_np = np.clip(final_output_np * 255, 0, 255).astype(np.uint8)

            # Print shape of images (optional for debugging)
            print("Rainy Input:", hf_input_np.shape)
            print("Ground Truth:", clean_hf_target_np.shape)
            print("Predicted:", predicted_hf_np.shape)

            # Plot them side by side
            fig, axs = plt.subplots(1, 3, figsize=(15, 5))

            axs[0].imshow(hf_input_np[0])
            axs[0].set_title("Rainy Input")
            axs[0].axis('off')

            axs[1].imshow(clean_hf_target_np[0])
            axs[1].set_title("Ground Truth Clean")
            axs[1].axis('off')

            axs[2].imshow(final_output_np[0])
            axs[2].set_title(f"Predicted Clean - Batch {batch_idx+1}")
            axs[2].axis('off')

            plt.show()

            # Save figure
            save_path = f"./output_images/batch_{batch_idx+1:03d}.png"
            plt.savefig(save_path, bbox_inches='tight')
            plt.close(fig)  # Close the figure to save memory




# After 50 batches, apply scheduler based on epoch loss
avg_epoch_loss = epoch_loss / num_batches_to_run
scheduler.step(avg_epoch_loss)

print(f"✅ 70-Batch Avg Loss: {epoch_loss / num_batches_to_run:.4f}")


In [ ]:
###Running for all test dataset
import matplotlib.pyplot as plt
import numpy as np
import torch
import os

# Make sure output directory exists
os.makedirs("/content/drive/MyDrive/rainy-image-dataset/test/output", exist_ok=True)

model.eval()  # Set model to evaluation mode
test_loss = 0.0

print("\n▶️ Running on entire test set...")

with torch.no_grad():
    for batch_idx, (hf_input, clean_hf_target, lf_input) in enumerate(test_loader):
        hf_input = hf_input.to(device)
        clean_hf_target = clean_hf_target.to(device)
        lf_input = lf_input.to(device)

        hf_input = preprocess_rainy_input(hf_input)

        predicted_hf = model(hf_input) * 0.5

        # Match shapes
        min_h = min(predicted_hf.shape[2], clean_hf_target.shape[2], lf_input.shape[2])
        min_w = min(predicted_hf.shape[3], clean_hf_target.shape[3], lf_input.shape[3])
        predicted_hf = predicted_hf[:, :, :min_h, :min_w]
        clean_hf_target = clean_hf_target[:, :, :min_h, :min_w]
        lf_input = lf_input[:, :, :min_h, :min_w]

        final_output = predicted_hf + lf_input
        final_output = torch.clamp(final_output, 0.0, 1.0)

        clean_full_target = clean_hf_target + lf_input
        clean_full_target = torch.clamp(clean_full_target, 0.0, 1.0)

        # Calculate test loss (optional)
        loss_mse = mse_loss_fn(predicted_hf, clean_hf_target)
        loss_ssim = 1 - ssim_loss_fn(predicted_hf, clean_hf_target)
        loss_perceptual = perceptual_loss_fn(predicted_hf.detach(), clean_hf_target.detach())
        loss = loss_mse + 0.1 * loss_perceptual + 0.5 * loss_ssim
        test_loss += loss.item()

        # Convert tensors to numpy for visualization
        def tensor_to_image(tensor):
            img = tensor.cpu().numpy()
            img = np.transpose(img, (0, 2, 3, 1))
            img = np.clip(img * 255, 0, 255).astype(np.uint8)
            return img

        hf_input_np = tensor_to_image(hf_input)
        clean_hf_target_np = tensor_to_image(clean_hf_target)
        final_output_np = tensor_to_image(final_output)

        # Save image
        fig, axs = plt.subplots(1, 3, figsize=(15, 5))
        axs[0].imshow(hf_input_np[0])
        axs[0].set_title("Rainy Input")
        axs[0].axis('off')

        axs[1].imshow(clean_hf_target_np[0])
        axs[1].set_title("Ground Truth")
        axs[1].axis('off')

        axs[2].imshow(final_output_np[0])
        axs[2].set_title(f"Prediction Batch {batch_idx+1}")
        axs[2].axis('off')

        save_path = f"/content/drive/MyDrive/rainy-image-dataset/test/output{batch_idx+1:03d}.png"
        plt.savefig(save_path, bbox_inches='tight')
        plt.close(fig)

        # Get and print absolute path
        full_path = os.path.abspath(save_path)
        print(f"Image saved at: {full_path}")

# Print average loss on test set
print(f"✅ Finished test set evaluation. Avg Loss: {test_loss / len(test_loader):.4f}")



In [ ]:
####Evaluation and calculating psnr and ssim metrics
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

total_psnr = 0.0
total_ssim = 0.0
num_images = 0

for batch_idx, (hf_input, clean_hf_target, lf_input) in enumerate(test_loader):
    hf_input = hf_input.to(device)
    clean_hf_target = clean_hf_target.to(device)

    # Preprocess
    hf_input = preprocess_rainy_input(hf_input)

    with torch.no_grad():
        predicted_hf = model(hf_input) * 0.5
        min_h = min(predicted_hf.shape[2], clean_hf_target.shape[2], lf_input.shape[2])
        min_w = min(predicted_hf.shape[3], clean_hf_target.shape[3], lf_input.shape[3])
        predicted_hf = predicted_hf[:, :, :min_h, :min_w]
        clean_hf_target = clean_hf_target[:, :, :min_h, :min_w]
        lf_input = lf_input[:, :, :min_h, :min_w]

        final_output = torch.clamp(predicted_hf + lf_input, 0.0, 1.0)
        clean_full_target = torch.clamp(clean_hf_target + lf_input, 0.0, 1.0)

        # Convert to NumPy (single image in batch assumed)
        output_np = final_output[0].cpu().permute(1, 2, 0).numpy()
        target_np = clean_full_target[0].cpu().permute(1, 2, 0).numpy()

        output_np = (output_np * 255).astype(np.uint8)
        target_np = (target_np * 255).astype(np.uint8)

        # Calculate metrics
        psnr_val = psnr_metric(target_np, output_np, data_range=255)
        ssim_val = ssim_metric(target_np, output_np, channel_axis=-1)

        total_psnr += psnr_val
        total_ssim += ssim_val
        num_images += 1

        # Optional: print per image
        print(f"[Test Image {batch_idx+1}] PSNR: {psnr_val:.2f}, SSIM: {ssim_val:.4f}")

# Average metrics
avg_psnr = total_psnr / num_images
avg_ssim = total_ssim / num_images
print(f"\n📊 Average PSNR: {avg_psnr:.2f} dB")
print(f"📊 Average SSIM: {avg_ssim:.4f}")


# New section

################Hybrid Model Implementation###############


In [ ]:
#Importing libraries
from IPython.display import display
import os
import cv2
import torch
import torch.nn as nn
from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms


In [ ]:
#Mount to drive
import os
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
clean_hf_dir = '/content/drive/MyDrive/rainy-image-dataset/train/clean'
test_rain_dir='/content/drive/MyDrive/rainy-image-dataset/test/rainy'
test_clean_dir='/content/drive/MyDrive/rainy-image-dataset/test/clean'
hf_guided_dir = '/content/drive/MyDrive/HF_guided_all'
lf_guided_dir = '/content/drive/MyDrive/LF_guided_all'

print("\nRainy HF images:")
print(sorted(os.listdir(hf_guided_dir)))
print("\nRainy LF images:")
print(sorted(os.listdir(lf_guided_dir)))
print("\nTest Rain images:")
print(sorted(os.listdir(test_rain_dir)))
print("\nTest Clean images:")
print(sorted(os.listdir(test_clean_dir)))
print("\nClean images:")
print(sorted(os.listdir(clean_hf_dir)))



In [ ]:
# Count only files (not directories)
num_hf_files = len([f for f in os.listdir(hf_guided_dir) if os.path.isfile(os.path.join(hf_guided_dir, f))])
print(f"Number of files: {num_hf_files}")

num_lf_files = len([f for f in os.listdir(lf_guided_dir) if os.path.isfile(os.path.join(lf_guided_dir, f))])
print(f"Number of files: {num_lf_files}")

In [ ]:
##test- train split (80-20) for HR_guided_ all folder
from sklearn.model_selection import train_test_split
import os

hf_guided_dir = '/content/drive/MyDrive/HF_guided_all'
all_files = [os.path.join(hf_guided_dir, f) for f in os.listdir(hf_guided_dir) if os.path.isfile(os.path.join(hf_guided_dir, f))]

train_hf_files, test_hf_files = train_test_split(all_files, test_size=0.2, random_state=42)

print(f"Training files: {len(train_hf_files)}")
print(f"Testing files: {len(test_hf_files)}")


In [ ]:
##test- train split (80-20) for LF_guided_ all folder
from sklearn.model_selection import train_test_split
import os

lf_guided_dir = '/content/drive/MyDrive/LF_guided_all'
all_files = [os.path.join(lf_guided_dir, f) for f in os.listdir(lf_guided_dir) if os.path.isfile(os.path.join(lf_guided_dir, f))]

train_lf_files, test_lf_files = train_test_split(all_files, test_size=0.2, random_state=42)

print(f"Training files: {len(train_lf_files)}")
print(f"Testing files: {len(test_lf_files)}")

In [ ]:
##################Dataset for LF Guided & HF Guided Image Pairs
class RainDetailDataset(Dataset):
    def __init__(self, lf_guided_files, hf_guided_files, clean_hf_dir, size=(256, 256)):
        self.lf_guided_files= lf_guided_files
        self.hf_guided_files = hf_guided_files
        self.clean_hf_dir = clean_hf_dir
        self.names = [os.path.basename(f) for f in hf_guided_files]  # Extract filenames from hf_guided_files
        self.size = size


    def __len__(self):
        return len(self.hf_guided_files)

    def __getitem__(self, idx):
        name = self.names[idx]


        # Construct full paths (updated)
        lf_guided_path = [f for f in self.lf_guided_files if name in f][0] # find path in the list
        hf_guided_path = self.hf_guided_files[idx]  # get hf_guided_path from hf_guided_files
        clean_hf_path = os.path.join(self.clean_hf_dir, name)
        print(clean_hf_path)  # Debugging print statement

        # Load with cv2
        lf_guided = cv2.imread(lf_guided_path)
        hf_guided = cv2.imread(hf_guided_path)
        clean_hf = cv2.imread(clean_hf_path)

        # Resize images
        lf_guided = cv2.resize(lf_guided, self.size)
        hf_guided = cv2.resize(hf_guided, self.size)
        clean_hf = cv2.resize(clean_hf, self.size)

        lf_guided= cv2.cvtColor(lf_guided, cv2.COLOR_BGR2RGB)
        hf_guided = cv2.cvtColor(hf_guided, cv2.COLOR_BGR2RGB)
        clean_hf = cv2.cvtColor(clean_hf, cv2.COLOR_BGR2RGB)

        # To tensor
        lf_guided = torch.from_numpy(lf_guided.transpose((2, 0, 1))).float() / 255.0
        hf_guided = torch.from_numpy(hf_guided.transpose((2, 0, 1))).float() / 255.0
        clean_hf = torch.from_numpy(clean_hf.transpose((2, 0, 1))).float() / 255.0

        return hf_guided, clean_hf, lf_guided



In [ ]:
##Normalizing for clean and rainy dataset ###
class RainDataset(Dataset):
    def __init__(self, rainy_dir, clean_dir, size=(256, 256)):
        self.rainy_dir = rainy_dir
        self.clean_dir = clean_dir
        self.clean_images = os.listdir(clean_dir)
        self.size = size

    def __len__(self):
        return len(self.clean_images)

    def __getitem__(self, idx):
        clean_filename = self.clean_images[idx]
        clean_path = os.path.join(self.clean_dir, clean_filename)
        rainy_filename = f"{os.path.splitext(clean_filename)[0]}_1.jpg"
        rainy_path = os.path.join(self.rainy_dir, rainy_filename)

        # Read using cv2 and normalize
        clean = cv2.imread(clean_path)
        rainy = cv2.imread(rainy_path)
        clean = cv2.resize(clean, self.size)
        rainy = cv2.resize(rainy, self.size)
        clean = cv2.cvtColor(clean, cv2.COLOR_BGR2RGB)
        rainy = cv2.cvtColor(rainy, cv2.COLOR_BGR2RGB)

        # Normalize to [0,1] and convert to tensor
        clean = torch.from_numpy(clean.transpose((2, 0, 1))).float() / 255.0
        rainy = torch.from_numpy(rainy.transpose((2, 0, 1))).float() / 255.0

        return rainy, clean


In [ ]:
#Dataset and DataLoader
train_dataset = RainDetailDataset(train_lf_files, train_hf_files, clean_hf_dir)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [ ]:
class DerainProcessorCV2:
    def __init__(self, radius=15, eps=1e-6):
        self.radius = radius
        self.eps = eps

    def guided_filter(self, I, p, r, eps):
        return cv2.ximgproc.guidedFilter(
            guide=I, src=p, radius=r, eps=eps, dDepth=-1
        )

    def decompose(self, img_tensor):
        img_np = img_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
        img_np = np.clip(img_np, 0, 1).astype(np.float32)

        base = np.zeros_like(img_np)
        for c in range(3):
            base[:, :, c] = self.guided_filter(img_np[:, :, c], img_np[:, :, c], self.radius, self.eps)

        detail = img_np - base
        base = torch.from_numpy(base).permute(2, 0, 1).unsqueeze(0).float()
        detail = torch.from_numpy(detail).permute(2, 0, 1).unsqueeze(0).float()
        return base, detail

    def reconstruct(self, base, predicted_detail, enhance=True):
        if enhance:
            predicted_detail *= 2
        return torch.clamp(base + predicted_detail, 0, 1)


In [ ]:
###Derain net class###
import torch
import torch.nn as nn

class DerainNet(nn.Module):
    def __init__(self):
        super(DerainNet, self).__init__()

        # Encoder
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # Bottleneck
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # Decoder
        self.deconv1 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.deconv2 = nn.Sequential(
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Sigmoid()  # Because output should be in [0,1]
        )

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)
        x4 = self.deconv1(x3)
        out = self.deconv2(x4)
        return out


In [ ]:
!pip install pytorch-ssim

In [ ]:
##code for simple_ssim since we are not installing the library
import torch
import torch.nn.functional as F

def simple_ssim(img1, img2, window_size=11, size_average=True):
    # Constants for stability
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    mu1 = F.avg_pool2d(img1, window_size, stride=1, padding=window_size//2)
    mu2 = F.avg_pool2d(img2, window_size, stride=1, padding=window_size//2)

    mu1_sq = mu1 * mu1
    mu2_sq = mu2 * mu2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.avg_pool2d(img1 * img1, window_size, stride=1, padding=window_size//2) - mu1_sq
    sigma2_sq = F.avg_pool2d(img2 * img2, window_size, stride=1, padding=window_size//2) - mu2_sq
    sigma12 = F.avg_pool2d(img1 * img2, window_size, stride=1, padding=window_size//2) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

    if size_average:
        return ssim_map.mean()
    else:
        return ssim_map


In [ ]:
####Preprocessing step -- to remove noise in the input image
import torchvision.transforms as T

# Simple denoising / smoothing
def preprocess_rainy_input(rainy_input_batch):
    """
    rainy_input_batch: torch tensor (batch_size, 3, 256, 256)
    Applies light Gaussian blur to reduce noise.
    """
    blur = T.GaussianBlur(kernel_size=5, sigma=(0.5, 1.5))
    return blur(rainy_input_batch)


In [ ]:
# Define Combined Loss
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.8):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        #self.ssim = pytorch_ssim.SSIM()

    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        #ssim_loss = 1 - ssim(pred.cpu().detach().numpy(), target.cpu().detach().numpy())  #Convert to numpy for SSIM computation
        #ssim_loss = 1 - torch_ssim(pred, target, data_range=1.0, size_average=True)
        ssim_loss = 1 - simple_ssim(pred, target)
        return self.alpha * mse_loss + (1 - self.alpha) * ssim_loss

# Set up
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DerainNet().to(device)
criterion = CombinedLoss(alpha=0.8)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)


In [ ]:
##installing libraries for perceptual loss
!pip install torchmetrics
!pip install pytorch-msssim

In [ ]:
##Defining all loss before training loop
import torch.nn as nn
import torch.nn.functional as F
from pytorch_msssim import ssim  # If using pytorch_msssim package
from torchvision import models

# MSE loss
mse_loss_fn = nn.MSELoss()

# SSIM loss (using pytorch_msssim)
def ssim_loss_fn(pred, target):
    return ssim(pred, target, data_range=1.0, size_average=True)

# Perceptual loss (based on VGG16)
class PerceptualLoss(nn.Module):
    def __init__(self, resize=True):
        super(PerceptualLoss, self).__init__()
        vgg = models.vgg16(pretrained=True).features[:16].eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg
        self.resize = resize

    def forward(self, pred, target):
        if self.resize:
            pred = F.interpolate(pred, size=(224, 224), mode='bilinear', align_corners=False)
            target = F.interpolate(target, size=(224, 224), mode='bilinear', align_corners=False)
        pred_features = self.vgg(pred)
        target_features = self.vgg(target)
        return F.l1_loss(pred_features, target_features)

perceptual_loss_fn = PerceptualLoss().to(device)


In [ ]:
####Train-test Split####
# For HF
hf_guided_dir = '/content/drive/MyDrive/HF_guided_all'
hf_guided = [os.path.join(hf_guided_dir, f) for f in os.listdir(hf_guided_dir) if os.path.isfile(os.path.join(hf_guided_dir, f))]
train_hf_guided, test_hf_guided = train_test_split(hf_guided, test_size=0.2, random_state=42)

# For LF
lf_guided_dir = '/content/drive/MyDrive/LF_guided_all'
lf_guided = [os.path.join(lf_guided_dir, f) for f in os.listdir(lf_guided_dir) if os.path.isfile(os.path.join(lf_guided_dir, f))]
train_lf_guided, test_lf_guided = train_test_split(lf_guided, test_size=0.2, random_state=42)


In [ ]:
###Dataset Class###
class GuidedImageDataset(Dataset):
    def __init__(self, lf_guided_files, hf_guided_files, transform=None):
        self.lf_guided_files = lf_guided_files
        self.hf_guided_files = hf_guided_files
        self.transform = transform

    def __len__(self):
        return len(self.hf_guided_files)

    def __getitem__(self, idx):
        lf_img = Image.open(self.lf_guided_files[idx]).convert("RGB")
        hf_img = Image.open(self.hf_guided_files[idx]).convert("RGB")

        if self.transform:
            lf_img = self.transform(lf_img)
            hf_img = self.transform(hf_img)

        return lf_img, hf_img  # LF = input, HF = target


In [ ]:
 ###Transformations and DataLoaders
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# Training and testing datasets
train_dataset = GuidedImageDataset(train_lf_guided, train_hf_guided, transform=transform)
test_dataset = GuidedImageDataset(test_lf_guided, test_hf_guided, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


In [ ]:
####Model Definition###
###Derain net class###
import torch
import torch.nn as nn

class DerainNet(nn.Module):
    def __init__(self):
        super(DerainNet, self).__init__()

        # Encoder
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # Bottleneck
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # Decoder
        self.deconv1 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.deconv2 = nn.Sequential(
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Sigmoid()  # Because output should be in [0,1]
        )

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)
        x4 = self.deconv1(x3)
        out = self.deconv2(x4)
        out = self.deconv2(x4 + x1)
        return out



In [ ]:
!pip install pytorch-ssim

In [ ]:
##code for simple_ssim since we are not installing the library
import torch
import torch.nn.functional as F

def simple_ssim(img1, img2, window_size=11, size_average=True):
    # Constants for stability
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    mu1 = F.avg_pool2d(img1, window_size, stride=1, padding=window_size//2)
    mu2 = F.avg_pool2d(img2, window_size, stride=1, padding=window_size//2)

    mu1_sq = mu1 * mu1
    mu2_sq = mu2 * mu2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.avg_pool2d(img1 * img1, window_size, stride=1, padding=window_size//2) - mu1_sq
    sigma2_sq = F.avg_pool2d(img2 * img2, window_size, stride=1, padding=window_size//2) - mu2_sq
    sigma12 = F.avg_pool2d(img1 * img2, window_size, stride=1, padding=window_size//2) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

    if size_average:
        return ssim_map.mean()
    else:
        return ssim_map


In [ ]:
####Preprocessing step -- to remove noise in the input image
import torchvision.transforms as T

# Simple denoising / smoothing
def preprocess_rainy_input(rainy_input_batch):
    """
    rainy_input_batch: torch tensor (batch_size, 3, 256, 256)
    Applies light Gaussian blur to reduce noise.
    """
    blur = T.GaussianBlur(kernel_size=5, sigma=(0.5, 1.5))
    return blur(rainy_input_batch)


In [ ]:
# Define Combined Loss
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.8):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        #self.ssim = pytorch_ssim.SSIM()

    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        #ssim_loss = 1 - ssim(pred.cpu().detach().numpy(), target.cpu().detach().numpy())  #Convert to numpy for SSIM computation
        #ssim_loss = 1 - torch_ssim(pred, target, data_range=1.0, size_average=True)
        ssim_loss = 1 - simple_ssim(pred, target)
        return self.alpha * mse_loss + (1 - self.alpha) * ssim_loss

# Set up
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DerainNet().to(device)
criterion = CombinedLoss(alpha=0.8)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)


In [ ]:
##installing libraries for perceptual loss
!pip install torchmetrics
!pip install pytorch-msssim

In [ ]:
##Defining all loss before training loop
import torch.nn as nn
import torch.nn.functional as F
from pytorch_msssim import ssim  # If using pytorch_msssim package
from torchvision import models

# MSE loss
mse_loss_fn = nn.MSELoss()

# SSIM loss (using pytorch_msssim)
def ssim_loss_fn(pred, target):
    return ssim(pred, target, data_range=1.0, size_average=True)

# Perceptual loss (based on VGG16)
class PerceptualLoss(nn.Module):
    def __init__(self, resize=True):
        super(PerceptualLoss, self).__init__()
        vgg = models.vgg16(pretrained=True).features[:16].eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg
        self.resize = resize

    def forward(self, pred, target):
        if self.resize:
            pred = F.interpolate(pred, size=(224, 224), mode='bilinear', align_corners=False)
            target = F.interpolate(target, size=(224, 224), mode='bilinear', align_corners=False)
        pred_features = self.vgg(pred)
        target_features = self.vgg(target)
        return F.l1_loss(pred_features, target_features)

perceptual_loss_fn = PerceptualLoss().to(device)


In [ ]:
####Training Loss
def tv_loss(img):
    """
    Computes total variation loss for an image.
    Args:
        img (Tensor): Shape (B, C, H, W)
    Returns:
        Total variation loss scalar
    """
    batch_size = img.size(0)
    h_tv = torch.mean(torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]))
    w_tv = torch.mean(torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]))
    return (h_tv + w_tv) / batch_size


In [ ]:
####Training Loop
import torch
import os
import numpy as np
import matplotlib.pyplot as plt

# Set up training for 70 batches
num_batches_to_run = 70

# Create the output_images directory if it doesn't exist, for saving the output images
os.makedirs("/content/drive/MyDrive/rainy-image-dataset/test/output_hybrid", exist_ok=True)

# Training loop
model.train()
epoch_loss = 0.0
print("\n▶️  Running 70 Batches Only")

alpha = 0.8  # You can tune this between 0.8 - 0.9

for batch_idx, (lf_input, clean_hf_target) in enumerate(train_loader):
    if batch_idx >= num_batches_to_run:
        print("✅ Reached 70 batches.")
        break

    lf_input = lf_input.to(device)
    clean_hf_target = clean_hf_target.to(device)

    # Forward pass
    predicted_hf = model(lf_input) * alpha  # [5] scaled down prediction

    # Match shapes if needed (ensuring that both predicted and target images are the same size)
    min_h = min(predicted_hf.shape[2], clean_hf_target.shape[2], lf_input.shape[2])
    min_w = min(predicted_hf.shape[3], clean_hf_target.shape[3], lf_input.shape[3])
    predicted_hf = predicted_hf[:, :, :min_h, :min_w]
    clean_hf_target = clean_hf_target[:, :, :min_h, :min_w]
    lf_input = lf_input[:, :, :min_h, :min_w]

    # Combine predicted HF and LF to get final output
    final_output = predicted_hf + lf_input  # No alpha scaling needed
    final_output = torch.clamp(final_output, 0.0, 1.0)  # Ensure pixel values are between 0 and 1

    # Prepare clean full image (target) for loss
    clean_full_target = clean_hf_target + lf_input
    clean_full_target = torch.clamp(clean_full_target, 0.0, 1.0)

    # Ensure correct input naming and detachment of tensors
    clean_hf = clean_hf_target  # Just aliasing for clarity

    # Loss functions
    loss_mse = mse_loss_fn(predicted_hf, clean_hf)
    loss_ssim = 1 - ssim_loss_fn(predicted_hf, clean_hf)
    loss_perceptual = perceptual_loss_fn(predicted_hf.detach(), clean_hf.detach())
    loss_tv = tv_loss(predicted_hf)  # or tv_loss(final_output) depending on what you want to regularize


    # Final combined loss
    loss = loss_mse + 0.1 * loss_perceptual + 0.5 * loss_ssim + 0.0001 * loss_tv
#    loss = loss_mse + 0.3 * loss_perceptual + 0.7 * loss_ssim

    # Backpropagation
    optimizer.zero_grad()
    #loss.backward(retain_graph=True)
    optimizer.step()

    epoch_loss += loss.item()

    # Optional progress print
    if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == num_batches_to_run:
        print(f"  Batch {batch_idx+1}/{num_batches_to_run} - Loss: {loss.item():.4f}")

    # Visualization after processing each batch
    if (batch_idx + 1) % 10 == 0:
        with torch.no_grad():
            # Prepare lf_input (Rainy input)
            lf_input_np = lf_input.cpu().detach().numpy()
            lf_input_np = np.transpose(lf_input_np, (0, 2, 3, 1))
            lf_input_np = np.clip(lf_input_np * 255, 0, 255).astype(np.uint8)

            # Prepare clean_hf_target (Ground truth clean)
            clean_hf_target_np = clean_hf_target.cpu().detach().numpy()
            clean_hf_target_np = np.transpose(clean_hf_target_np, (0, 2, 3, 1))
            clean_hf_target_np = np.clip(clean_hf_target_np * 255, 0, 255).astype(np.uint8)

            # Prepare predicted_hf (Predicted clean)
            predicted_hf_np = predicted_hf.cpu().detach().numpy()
            predicted_hf_np = np.transpose(predicted_hf_np, (0, 2, 3, 1))
            predicted_hf_np = np.clip(predicted_hf_np * 255, 0, 255).astype(np.uint8)

            # Predicted final output
            final_output_np = final_output.cpu().detach().numpy()
            final_output_np = np.transpose(final_output_np, (0, 2, 3, 1))
            final_output_np = np.clip(final_output_np * 255, 0, 255).astype(np.uint8)

            # Print shape of images (optional for debugging)
            print("Rainy Input:", lf_input_np.shape)
            print("Ground Truth:", clean_hf_target_np.shape)
            print("Predicted:", predicted_hf_np.shape)

            # Plot them side by side
            fig, axs = plt.subplots(1, 3, figsize=(15, 5))

            axs[0].imshow(lf_input_np[0])
            axs[0].set_title("Rainy Input")
            axs[0].axis('off')

            axs[1].imshow(clean_hf_target_np[0])
            axs[1].set_title("Ground Truth Clean")
            axs[1].axis('off')

            axs[2].imshow(final_output_np[0])
            axs[2].set_title(f"Predicted Clean - Batch {batch_idx+1}")
            axs[2].axis('off')

            plt.show()

            # Save figure
            save_path = f"/content/drive/MyDrive/rainy-image-dataset/test/output_hybrid{batch_idx+1:03d}.png"
            plt.savefig(save_path, bbox_inches='tight')
            plt.close(fig)  # Close the figure to save memory

# After 70 batches, apply scheduler based on epoch loss
avg_epoch_loss = epoch_loss / num_batches_to_run
scheduler.step(avg_epoch_loss)

print(f"✅ 70-Batch Avg Loss: {epoch_loss / num_batches_to_run:.4f}")


In [ ]:
###Evaluation###3
#### Evaluation and calculating PSNR and SSIM metrics

from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

total_psnr = 0.0
total_ssim = 0.0
num_images = 0

# Evaluate model on the test set
model.eval()  # Set model to evaluation mode
with torch.no_grad():  # Disable gradient calculation for evaluation
    for batch_idx, (lf_input, clean_hf_target) in enumerate(test_loader):

        lf_input = lf_input.to(device)
        clean_hf_target = clean_hf_target.to(device)

        # Forward pass (Prediction)
        predicted_hf = model(lf_input) * 0.8  # Scale down prediction as in training loop

        # Match the sizes
        min_h = min(predicted_hf.shape[2], clean_hf_target.shape[2], lf_input.shape[2])
        min_w = min(predicted_hf.shape[3], clean_hf_target.shape[3], lf_input.shape[3])
        predicted_hf = predicted_hf[:, :, :min_h, :min_w]
        clean_hf_target = clean_hf_target[:, :, :min_h, :min_w]
        lf_input = lf_input[:, :, :min_h, :min_w]

        # Combine predicted HF and LF to get final output
        final_output = predicted_hf + lf_input
        final_output = torch.clamp(final_output, 0.0, 1.0)

        # Clean full target (ground truth)
        clean_full_target = clean_hf_target + lf_input
        clean_full_target = torch.clamp(clean_full_target, 0.0, 1.0)

        # Convert tensor to numpy for PSNR and SSIM calculation
        output_np = final_output[0].cpu().permute(1, 2, 0).numpy()
        target_np = clean_full_target[0].cpu().permute(1, 2, 0).numpy()

        # Convert from [0,1] to [0,255] for PSNR/SSIM calculation
        output_np = (output_np * 255).astype(np.uint8)
        target_np = (target_np * 255).astype(np.uint8)

        # Calculate PSNR and SSIM
        psnr_val = psnr_metric(target_np, output_np, data_range=255)
        ssim_val = ssim_metric(target_np, output_np, channel_axis=-1)

        # Accumulate metrics
        total_psnr += psnr_val
        total_ssim += ssim_val
        num_images += 1

        # Optional: Print per image
        print(f"[Test Image {batch_idx + 1}] PSNR: {psnr_val:.2f}, SSIM: {ssim_val:.4f}")

# Calculate average metrics
avg_psnr = total_psnr / num_images
avg_ssim = total_ssim / num_images

# Print final evaluation results
print(f"\n📊 Average PSNR: {avg_psnr:.2f} dB")
print(f"📊 Average SSIM: {avg_ssim:.4f}")


In [ ]:
# Training loop for Residual DerainNet
import torch
import os
import numpy as np
import matplotlib.pyplot as plt

# Create output directory
os.makedirs("./output_images", exist_ok=True)

# Hyperparameters
num_batches_to_run = 70
alpha = 0.8

model.train()
epoch_loss = 0.0
print("\n▶️  Running 70 Batches Only")

for batch_idx, (lf_input, clean_hf_target) in enumerate(train_loader):
    if batch_idx >= num_batches_to_run:
        print("✅ Reached 70 batches.")
        break

    lf_input = lf_input.to(device)
    clean_hf_target = clean_hf_target.to(device)

    # Ground truth full clean image
    clean_full_target = lf_input + clean_hf_target
    clean_full_target = torch.clamp(clean_full_target, 0.0, 1.0)

    # Forward pass: model predicts rain, subtracts from input
    final_output, rain_pred = model(lf_input)

    # Clamp predicted clean image
    final_output = torch.clamp(final_output, 0.0, 1.0)

    # Match shapes
    min_h = min(final_output.shape[2], clean_full_target.shape[2])
    min_w = min(final_output.shape[3], clean_full_target.shape[3])
    final_output = final_output[:, :, :min_h, :min_w]
    clean_full_target = clean_full_target[:, :, :min_h, :min_w]

    # Losses
    loss_mse = mse_loss_fn(final_output, clean_full_target)
    loss_ssim = 1 - ssim_loss_fn(final_output, clean_full_target)
    loss_perceptual = perceptual_loss_fn(final_output.detach(), clean_full_target.detach())
    loss_tv = tv_loss(final_output)

    # Combine losses
    loss = loss_mse + 0.1 * loss_perceptual + 0.5 * loss_ssim + 0.0001 * loss_tv

    # Backprop
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

    # Print progress
    if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == num_batches_to_run:
        print(f"  Batch {batch_idx+1}/{num_batches_to_run} - Loss: {loss.item():.4f}")

        # Visualization
        with torch.no_grad():
            # Detach and move to CPU
            rainy_input = lf_input.cpu().detach().numpy()
            ground_truth = clean_full_target.cpu().detach().numpy()
            predicted_clean = final_output.cpu().detach().numpy()

            # Convert to HWC and uint8
            rainy_input = np.transpose(rainy_input, (0, 2, 3, 1))
            ground_truth = np.transpose(ground_truth, (0, 2, 3, 1))
            predicted_clean = np.transpose(predicted_clean, (0, 2, 3, 1))

            rainy_input = np.clip(rainy_input * 255, 0, 255).astype(np.uint8)
            ground_truth = np.clip(ground_truth * 255, 0, 255).astype(np.uint8)
            predicted_clean = np.clip(predicted_clean * 255, 0, 255).astype(np.uint8)

            # Plot
            fig, axs = plt.subplots(1, 3, figsize=(15, 5))
            axs[0].imshow(rainy_input[0])
            axs[0].set_title("Rainy Input")
            axs[0].axis('off')

            axs[1].imshow(ground_truth[0])
            axs[1].set_title("Ground Truth Clean")
            axs[1].axis('off')

            axs[2].imshow(predicted_clean[0])
            axs[2].set_title(f"Predicted Clean - Batch {batch_idx+1}")
            axs[2].axis('off')
            plt.show()

# Learning rate scheduling
avg_epoch_loss = epoch_loss / num_batches_to_run
scheduler.step(avg_epoch_loss)
print(f"✅ 70-Batch Avg Loss: {avg_epoch_loss:.4f}")


In [ ]:
import torch
import os
import numpy as np
import matplotlib.pyplot as plt

# Create output directory
os.makedirs("./output_images", exist_ok=True)

# Hyperparameters
num_batches_to_run = 70
alpha = 0.8

# Ensure model is in training mode
model.train()
epoch_loss = 0.0
print("\n▶️  Running 70 Batches Only")

# Loop through the data batches
for batch_idx, (lf_input, clean_hf_target) in enumerate(train_loader):
    if batch_idx >= num_batches_to_run:
        print("✅ Reached 70 batches.")
        break

    # Move inputs to the device (GPU or CPU)
    lf_input = lf_input.to(device)
    clean_hf_target = clean_hf_target.to(device)

    # Ground truth full clean image (sum of low-frequency input and high-frequency target)
    clean_full_target = lf_input + clean_hf_target
    clean_full_target = torch.clamp(clean_full_target, 0.0, 1.0)

    outputs = model(lf_input)
    print(f"Model output shape: {outputs}")


    # Forward pass: model predicts rain, subtracts from input
    final_output, rain_pred = model(lf_input)

    # Clamp predicted clean image to [0, 1] range
    final_output = torch.clamp(final_output, 0.0, 1.0)

    # Ensure the shapes match
    min_h = min(final_output.shape[2], clean_full_target.shape[2])
    min_w = min(final_output.shape[3], clean_full_target.shape[3])
    final_output = final_output[:, :, :min_h, :min_w]
    clean_full_target = clean_full_target[:, :, :min_h, :min_w]

    # Losses
    loss_mse = mse_loss_fn(final_output, clean_full_target)
    loss_ssim = 1 - ssim_loss_fn(final_output, clean_full_target)
    loss_perceptual = perceptual_loss_fn(final_output.detach(), clean_full_target.detach())
    loss_tv = tv_loss(final_output)

    # Combine losses
    loss = loss_mse + 0.1 * loss_perceptual + 0.5 * loss_ssim + 0.0001 * loss_tv

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

    # Print progress and visualize every 10 batches or the final batch
    if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == num_batches_to_run:
        print(f"  Batch {batch_idx+1}/{num_batches_to_run} - Loss: {loss.item():.4f}")

        # Visualization: Detach and move tensors to CPU for plotting
        with torch.no_grad():
            rainy_input = lf_input.cpu().detach().numpy()
            ground_truth = clean_full_target.cpu().detach().numpy()
            predicted_clean = final_output.cpu().detach().numpy()

            # Convert from [B, C, H, W] to [B, H, W, C]
            rainy_input = np.transpose(rainy_input, (0, 2, 3, 1))
            ground_truth = np.transpose(ground_truth, (0, 2, 3, 1))
            predicted_clean = np.transpose(predicted_clean, (0, 2, 3, 1))

            # Clip and convert to uint8 for displaying
            rainy_input = np.clip(rainy_input * 255, 0, 255).astype(np.uint8)
            ground_truth = np.clip(ground_truth * 255, 0, 255).astype(np.uint8)
            predicted_clean = np.clip(predicted_clean * 255, 0, 255).astype(np.uint8)

            # Plotting
            fig, axs = plt.subplots(1, 3, figsize=(15, 5))
            axs[0].imshow(rainy_input[0])
            axs[0].set_title("Rainy Input")
            axs[0].axis('off')

            axs[1].imshow(ground_truth[0])
            axs[1].set_title("Ground Truth Clean")
            axs[1].axis('off')

            axs[2].imshow(predicted_clean[0])
            axs[2].set_title(f"Predicted Clean - Batch {batch_idx+1}")
            axs[2].axis('off')
            plt.show()

# Learning rate scheduling based on the average loss
avg_epoch_loss = epoch_loss / num_batches_to_run
scheduler.step(avg_epoch_loss)
print(f"✅ 70-Batch Avg Loss: {avg_epoch_loss:.4f}")
